<div align="center">

# **SWE 485**  
## Customer Churn Prediction Using Machine Learning

</div>

---
---

<div align="center">

# **Phase 2 - Part B**

</div>



<div align="center">

# **Generative AI**

</div>


---
---

<div align="center">

**Generative AI is a type of artificial intelligence that can create new content such as text, images, or code based on patterns it learned from data. Instead of only analyzing or predicting outcomes, it generates human-like responses. In this project the Generative AI will be used to transform model predictions into clear, understandable insights. Instead of only outputting whether a customer will stay or churn, it explains the reasoning behind the prediction, suggests business actions, classifies risk levels, and summarizes customer behavior.**

**This is important because raw model outputs (like "Will Churn") are not sufficient for decision-making. By using Generative AI, we convert these predictions into meaningful explanations and recommendations that help analysts and decision-makers understand the results and take appropriate actions.**

</div>





---

### **Introduction**

This Part focuses on integrating Generative AI into the customer churn prediction system. The goal is to enhance the model outputs by generating human-readable explanations, business recommendations, risk analysis, and customer summaries based on predicted outcomes.

### **Objective**

The objective of this phase is to use Generative AI to:
- Explain model predictions in a clear and simple way
- Provide actionable business recommendations
- Classify customer risk levels
- Generate concise customer summaries


---

### **Generative AI Model Selection & Setup**

In this project, we selected **Google Gemini (gemini-flash-latest)** as the generative AI model due to its fast response time, strong text generation capabilities, and ease of integration through a simple API. This makes it suitable for generating explanations, recommendations, risk assessments, and summaries based on model predictions.

To ensure secure and flexible usage, the API key is stored using environment variables instead of being hardcoded in the code. The `.env` file is used to manage sensitive information safely, following best practices for key handling.

The setup process includes:
- Loading environment variables using `dotenv`
- Retrieving the API key securely
- Initializing the Gemini client using the official SDK

This setup enables the system to communicate with the model and generate dynamic, context-aware responses based on customer data.


### **Implementation & API Integration Code**

The Generative AI functionality is implemented using Python and the Gemini API. The integration is designed to securely handle sensitive information while maintaining clean and reusable code.

The API key is stored in a `.env` file and accessed using the `dotenv` library. This prevents exposing the key in the source code or repository, as the system retrieves it at runtime to initialize the Gemini client.

A separate template file (`api_config_template.py`) is included to demonstrate the API configuration without revealing the actual key. This ensures that the project can be shared safely while following secure development practices.

In addition, the integration incorporates the trained machine learning model during runtime. The trained Random Forest pipeline is saved and loaded using the `joblib` library, which is a Python utility used for efficiently storing and retrieving machine learning models. This allows the model to be reused without retraining, improving efficiency and ensuring consistency in predictions.

The saved model file (`rf_pipeline.pkl`) is loaded and used to generate predictions directly from customer data.

Customer cases are selected from the preprocessed dataset to ensure compatibility with the model’s expected input structure. Each case is represented in two forms: a complete feature set used for prediction, and a simplified subset used to generate clear and human-readable explanations.

The simplified subset is composed of a limited number of features that are most relevant and interpretable for explaining customer behavior in a banking context. These features capture key aspects of customer behavior and financial status:

- **CreditScore** reflects financial reliability and risk level  
- **Age** represents customer stability and life stage  
- **Balance** indicates how actively the account is used  
- **NumOfProducts** shows the strength of the relationship with the bank  
- **IsActiveMember** captures customer engagement  
- **EstimatedSalary** reflects financial capacity and potential value  

These features are selected because they provide meaningful insights that can be easily understood and explained using Generative AI. Using all dataset features (such as IDs or encoded variables) does not improve explanation quality and may reduce clarity.

This design maintains a balance between model relevance, interpretability, and clear communication of results.

Overall, this implementation enables seamless integration between the machine learning model and Generative AI, ensuring secure key handling, consistent predictions, and reliable outputs.

In [14]:
import joblib

rf_model = joblib.load("rf_pipeline.pkl")
print("Model loaded!")

Model loaded!


In [68]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

gemini_api_key = os.getenv("KEY")
client = genai.Client(api_key=gemini_api_key)

---

### **Testing Framework & Output Comparison**

To evaluate the effectiveness of the designed prompts, a structured testing framework was implemented using multiple customer profiles from the dataset.

#### **Testing Framework**

Each prompt template was tested using  **3 different customer cases**, selected to represent diverse scenarios such as:
- Customers predicted to stay
- Customers predicted to churn
- Customers with mixed characteristics

For each case:
- Customer data was selected directly from the preprocessed dataset  
- The trained machine learning model was used to generate the prediction  
- A subset of interpretable features was extracted for explanation purposes  
- The prompt template was filled using the customer features and the model prediction
- The response was obtained using the Gemini API  

This approach ensures that all generated outputs are based on actual model predictions and reflect realistic customer behavior rather than predefined labels.

#### **Output Comparison**

The generated outputs were evaluated based on the following criteria:

- **Relevance:**  
  Whether the response aligns with the model prediction (Stay/Churn)

- **Detail & Completeness:**  
  Whether the response includes sufficient explanation or actionable insights

- **Clarity & Readability:**  
  Whether the output is easy to understand for non-technical users

- **Personalization:**  
  Whether the response reflects the specific customer data instead of being generic

- **Consistency:**  
  Whether similar cases produce logically consistent outputs

Additionally, a simple quantitative comparison was performed based on:
- Response length  
- Use of domain-specific keywords  
- Readability level  

This framework allows systematic evaluation of all prompt templates and supports selecting the most effective one.


In [24]:
import pandas as pd

df = pd.read_csv("Dataset/Preprocessed_Churn_Modelling_Data.csv")


In [65]:
def get_prediction(model_features):
    import pandas as pd

    df_input = pd.DataFrame([model_features])
    pred = rf_model.predict(df_input)[0]

    return "Will Stay" if pred == 0 else "Will Churn"


def build_case(row):
    # full features for model
    model_features = row.drop("Exited").to_dict()

    # readable subset for prompt
    prompt_features = {
    "CreditScore": int(row["CreditScore"]),
    "Age": int(row["Age"]),
    "Balance": float(row["Balance"]),
    "NumOfProducts": int(row["NumOfProducts"]),
    "IsActiveMember": int(row["IsActiveMember"]),
    "EstimatedSalary": float(row["EstimatedSalary"])
}

    return {
        "features": prompt_features,
        "prediction": get_prediction(model_features)
    }


# select 3 cases from dataset
case1 = build_case(df.iloc[0])
case2 = build_case(df.iloc[11])
case3 = build_case(df.iloc[6001])

cases = [case1, case2, case3]

print(case1)
print(case2)
print(case3)

{'features': {'CreditScore': 619, 'Age': 42, 'Balance': 0.0, 'NumOfProducts': 1, 'IsActiveMember': 1, 'EstimatedSalary': 101348.88}, 'prediction': 'Will Stay'}
{'features': {'CreditScore': 497, 'Age': 24, 'Balance': 0.0, 'NumOfProducts': 2, 'IsActiveMember': 0, 'EstimatedSalary': 76390.01}, 'prediction': 'Will Churn'}
{'features': {'CreditScore': 775, 'Age': 30, 'Balance': 0.0, 'NumOfProducts': 1, 'IsActiveMember': 0, 'EstimatedSalary': 193880.6}, 'prediction': 'Will Stay'}


---

## **Prompt 1: Explanation of Prediction**

In [59]:
def create_prompt(case):
    return f"""
You are a banking analyst.

Customer data:
{case['features']}

Prediction: {case['prediction']}

Explain this prediction using ONLY the provided features.

Rules:
- Use exactly 4 bullet points
- Be short and clear
- Do not mention any feature that is not provided
- Do not make assumptions about customer intentions
- Do not use vague phrases like "likely target" or "typically"
- Keep each bullet tied to one visible feature
- make sure the explanation consistently supports the prediction and does not contradict it in any way
"""



In [60]:
for i, case in enumerate(cases, 1):
    print(f"\n===== Case {i} =====")

    prompt = create_prompt(case)

    response = client.models.generate_content(
        model="models/gemini-flash-latest",
        contents=prompt
    )

    print(response.text)


===== Case 1 =====


Based on the provided data, the prediction that the customer will stay is supported by these factors:

*   The active member status confirms current engagement with bank services.
*   An estimated salary of $101,348.88 establishes a solid financial foundation for the account.
*   The customer's age of 42 years places them in a stable demographic for banking.
*   A credit score of 619 meets the necessary threshold for maintaining the relationship.

===== Case 2 =====


*   The credit score of 497 is a low, high-risk value.
*   The account balance of 0.0 shows zero financial commitment.
*   The inactive member status (0) indicates a lack of engagement.
*   The age of 24 is a high-risk indicator for account closure.

===== Case 3 =====


As a banking analyst, I have identified the following factors that support the prediction that the customer will stay:

*   The Credit Score of 775 indicates a high level of financial reliability and account stability.
*   The Estimated Salary of 193880.6 provides a substantial financial foundation for the customer to maintain their relationship with the bank.
*   The Age of 30 represents a stage in the customer lifecycle with a long remaining duration for banking services.
*   The count of one product establishes a formal and existing service relationship with the institution.




#### **Qualitative Analysis**

The generated explanations were evaluated based on clarity, relevance, consistency, and overall usefulness across different customer cases.

- **Relevance:**  
  The explanations generally align well with the model predictions. In churn case the responses highlight low engagement and weaker financial indicators, while in stay cases they emphasize stability and active behavior.  
  However, in some cases the reasoning appears slightly simplified and does not fully capture complex relationships between features.

- **Detail & Completeness:**  
  The explanations provide a thorough breakdown of the prediction by highlighting multiple relevant features such as credit score, age, balance, and engagement. This gives a clear understanding of why the model made its decision.  
  However, some explanations remain at a general level and do not fully explore complex interactions between features.

- **Clarity & Readability:**  
  The outputs are clear and easy to understand, using simple language suitable for non-technical users. The use of bullet points improves readability.  

- **Personalization:**  
  The outputs reference actual customer values ( age, salary, balance) making them personalized.  
  Despite this, some explanations still appear partially generic and do not fully adapt to unique edge cases.

- **Safety & Factual Accuracy:**  
  The explanations are mostly grounded in the provided features and model prediction, reducing the risk of hallucinations.  
  However, some statements slightly generalize customer behavior (e.g., assumptions about financial habits), which introduces minor risk of over-interpretation.




#### **Quantitative Analysis**

A simple quantitative evaluation was conducted to assess structure and readability.

- **Response Length:**  
  Each response contains approximately 60–100 words, providing sufficient detail while remaining concise.  


- **Keyword Alignment:**  
  The responses consistently use domain-specific terms such as *credit score*, *balance*, and *engagement*.  

- **Readability Score:** Flesch Reading Ease ≈ 58, Flesch-Kincaid Grade ≈ 9  
  The outputs use moderately complex language due to detailed explanations and longer sentences. This supports better reasoning and justification.  
  However, the increased detail slightly reduces readability for non-technical users.

- **User Preference Simulation:**  
  Users who prefer detailed reasoning and deeper understanding would favor this prompt, as it explains why the prediction was made.  
  However, users looking for quick insights or business actions may find it too long or analytical.



#### **Overall Observation**

The prompt produces explanations that are clear, consistent, and aligned with model predictions. While the outputs are effective and easy to understand, they tend to be somewhat repetitive and may not fully capture complex or unique customer scenarios. This highlights a trade-off between simplicity and depth in the generated explanations.

---

## **Prompt 2: Recommendations**

In [61]:
def create_recommendation_prompt(case):
    return f"""
You are a banking analyst.

Customer data:
{case['features']}

Prediction: {case['prediction']}

Task:
Suggest what the bank should do for this customer.

Rules:
- Give exactly 3 recommendations
- Keep each recommendation short and actionable
- Base every recommendation only on the provided features and prediction
- Do not mention features that are not shown
- Do not make assumptions about hidden customer intentions
- Focus on retention, engagement, or product strategy
- Write in simple professional language
"""


In [62]:
for i, case in enumerate(cases, 1):
    print(f"\n===== Recommendation Case {i} =====")

    prompt = create_recommendation_prompt(case)

    response = client.models.generate_content(
        model="models/gemini-flash-latest",
        contents=prompt
    )

    print(response.text)


===== Recommendation Case 1 =====


1. **Incentivize Deposits**: Offer a promotional interest rate or a cash-back reward for setting up a direct salary deposit to address the current 0.0 balance.

2. **Cross-Sell a Second Product**: Introduce a credit card or a low-interest personal loan to increase the number of products from one, deepening the customer's institutional loyalty.

3. **Provide Credit Improvement Tools**: Offer personalized financial counseling or credit-monitoring services to help the customer manage their $101,348 salary and improve their 619 credit score.

===== Recommendation Case 2 =====


1. **Offer a direct deposit incentive:** Encourage the customer to deposit their salary into their account to address the zero-balance status and increase account activity.

2. **Provide credit-building resources:** Offer personalized financial tools or a secured credit product to help the customer improve their low credit score.

3. **Launch a re-engagement campaign:** Send targeted communications or fee-waiver offers to activate the customer's two existing products and transition them to active member status.

===== Recommendation Case 3 =====


1. **Incentivize account funding:** Offer a high-yield savings promotion to encourage the customer to deposit their high estimated salary into their currently empty account.

2. **Cross-sell premium products:** Leverage the excellent credit score by offering a pre-approved premium credit card to increase the number of products held beyond just one.

3. **Re-activate membership:** Targeted outreach with a loyalty reward or fee waiver to convert the customer into an active member and improve engagement levels.



#### **Qualitative Analysis**

- **Relevance:**  
  The recommendations are strongly aligned with both the customer data and the model prediction. Each suggestion reflects key features such as balance, credit score, activity status, and number of products.  
  However, some recommendations rely on general patterns and do not fully capture more complex relationships between multiple features.

- **Detail & Completeness:**  
  The responses provide sufficient detail by offering exactly three actionable recommendations per case. Each recommendation includes a clear action making the output useful and complete.  
  On the other hand the depth of explanation is sometimes limited as the reasoning behind each recommendation is not always fully elaborated.

- **Clarity & Readability:**  
  The recommendations are easy to understand and well structured using a numbered format. The language is simple and suitable for non technical users.  

- **Personalization:**  
  The output is personalized to each customer case, with recommendations adapting to differences in features such as balance, engagement, and number of products.  

- **Safety & Factual Accuracy:**  
  The recommendations are generally based on the provided features and follow realistic banking practices.  
  However, some suggestions involve implicit assumptions ( how the customer uses their salary), which may introduce mild hallucination risk.

#### **Quantitative Analysis**

- **Response Length:**  
  Each case consistently produces exactly three recommendations, ensuring uniformity across outputs.  

- **Keyword Usage:**  
  The responses include relevant domain-specific terms such as “deposit,” “credit,” “engagement,” “cross-sell,” and “loyalty.”  
  Some keywords are reused frequently leading to slight redundancy.

- **Readability Level:** Flesch Reading Ease ≈ 70, Flesch-Kincaid Grade ≈ 7  
  The outputs use clear and simple language, making them easy to understand for non-technical users and business stakeholders.  
  However, the simplicity may limit deeper analytical insight in some recommendations.

- **User Preference Simulation:**  
  Business users and decision-makers would prefer this prompt, as it provides clear and actionable steps that can be directly implemented.  
  However, users who want to understand the reasoning behind decisions may find it lacking in depth. 


#### **Overall Observation**

The recommendation prompt produces outputs that are relevant, actionable, and well-structured. The responses are clearly aligned with customer data and model predictions, making them practical for real-world use. The consistent format and simple language improve usability and readability.

However, the outputs show noticeable repetition in strategy patterns and moderate personalization in some cases. Additionally, the simplicity of the responses may limit deeper insights. Despite these limitations, the prompt remains effective in generating meaningful and business-oriented recommendations.

---

## **Prompt 3: Risk Classification**

In [63]:
def create_risk_prompt(case):
    return f"""
You are a banking risk analyst.

Customer data:
{case['features']}

Prediction: {case['prediction']}

Task:
Classify this customer into one of the following:
- Low risk
- Medium risk
- High risk

Rules:
- If prediction is "Will Churn", the risk must be Medium or High
- If prediction is "Will Stay", the risk must be Low or Medium
- Base the decision only on the provided features
- Do not assume any missing information

Output format:
- Start with the risk level (Low / Medium / High)
- Then give up to 4 short bullet points explaining why
- Each point must reference the given features
- Keep it simple and concise
"""

In [64]:
for i, case in enumerate(cases, 1):
    print(f"\n===== Risk Analysis Case {i} =====")

    prompt = create_risk_prompt(case)

    response = client.models.generate_content(
        model="models/gemini-flash-latest",
        contents=prompt
    )

    print(response.text)


===== Risk Analysis Case 1 =====


Medium risk

* **Fair Credit Score:** A score of 619 is at the lower end of the "fair" range, indicating some credit vulnerability.
* **Zero Balance:** Having 0.0 in the account reduces the customer's financial ties and "stickiness" to the bank.
* **Single Product:** Usage of only one product makes the customer more susceptible to competitor offers than those with multiple products.
* **Active Status:** Being an active member with a salary over $100k balances the risks, supporting the "Will Stay" prediction.

===== Risk Analysis Case 2 =====


High risk

* **Poor Credit Score:** A score of 497 is significantly low, indicating higher financial risk and lower customer stickiness.
* **Zero Account Balance:** A $0.0 balance suggests no financial depth or commitment to the institution.
* **Inactive Status:** Being an inactive member (IsActiveMember: 0) is a strong leading indicator of imminent churn.
* **Low Engagement:** Despite having 2 products, the lack of activity and zero balance makes the customer a high flight risk.

===== Risk Analysis Case 3 =====


Medium risk

* High Credit Score (775) and high Estimated Salary ($193,880.60) indicate financial stability and low default risk.
* IsActiveMember status is 0, showing a lack of engagement with the bank's services.
* A Balance of 0.0 suggests the account is not being used for primary banking needs.
* The customer only holds 1 product, which historically increases the likelihood of future attrition.



#### **Qualitative Analysis**

- **Relevance:**  
  The risk classifications are partially aligned with the customer data and model predictions. For example, high-risk profiles (low credit score, zero balance, inactivity) are correctly identified as high risk.  
  However, there is a clear inconsistency when a customer predicted as **"Will Stay"** is classified as **medium risk**, which contradicts the expected logic and the prompt. This indicates that the model does not strictly follow the intended mapping between prediction and risk level.

- **Detail & Completeness:**  
  The responses include a clear risk label followed by multiple supporting points. Key features such as credit score, balance, and engagement are referenced in the explanation.  
  However, the reasoning remains relatively surface level and does not deeply analyze how features interact to influence risk.

- **Clarity & Readability:**  
  The outputs are well-structured and easy to read. Starting with the risk level followed by bullet points improves clarity and makes the explanation accessible to non-technical users.  

- **Personalization:**  
  The explanations are based on the specific customer features, and different cases highlight different risk factors.  
  However, similar reasoning patterns (like zero balance, inactivity) appear across multiple cases, limiting the uniqueness of responses.

- **Safety & Factual Accuracy:**  
  The risk classifications and explanations are mostly tied to the given features, limiting hallucination.  
  However, there is inconsistency in how risk levels are assigned (e.g., a "Will Stay" customer classified as medium risk), which affects factual reliability.

#### **Quantitative Analysis**

- **Response Length:**  
  Each case produces a short response with a risk label and up to four bullet points, maintaining consistent length across outputs.

- **Keyword Usage:**  
  The responses consistently use relevant terms such as “credit score,” “balance,” “engagement,” and “risk,” indicating strong domain alignment.  

- **Readability Level:** Flesch Reading Ease ≈ 75, Flesch-Kincaid Grade ≈ 6  
  The outputs are very simple and concise, with short bullet points that improve readability and quick understanding.  
  However, the high simplicity reduces depth and detailed reasoning in the explanations.

- **User Preference Simulation:**  
  Users who want quick and simple summaries would prefer this prompt, as it provides a clear risk level with concise justification.  
  However, users who need detailed insights or strategic guidance may find it too basic.


#### **Overall Observation**

The risk classification prompt produces clear, structured, and easy to understand outputs. The explanations are grounded in customer features and provide reasonable justifications for the assigned risk levels.

However, there is a critical limitation in classification consistency, where the assigned risk level does not always align with the model prediction (a "Will Stay" customer classified as medium risk). 

Overall, the prompt is effective in generating simple risk explanations but requires stronger control over classification logic to ensure reliability.

---

## **Prompt 4: Customer Summary**

In [66]:
def create_summary_prompt(case):
    return f"""
You are a banking analyst.

Customer data:
{case['features']}

Prediction: {case['prediction']}

Task:
Write a short customer profile.

Rules:
- Write 2–3 sentences only
- Use simple and clear language
- Focus on behavior, financial habits, and engagement
- Base the summary only on the provided features and prediction
- Do not assume any missing information
- Keep sentences concise and consistent
"""

In [69]:
for i, case in enumerate(cases, 1):
    print(f"\n===== Customer Summary Case {i} =====")

    prompt = create_summary_prompt(case)

    response = client.models.generate_content(
        model="models/gemini-flash-latest",
        contents=prompt
    )

    print(response.text)


===== Customer Summary Case 1 =====


This 42-year-old active member earns a six-figure salary but currently maintains a zero-balance account and a moderate credit score. Despite utilizing only one financial product, their consistent engagement suggests they will remain loyal to the bank.

===== Customer Summary Case 2 =====


This young individual maintains a zero balance and a low credit score despite earning a steady salary. Although they use two banking products, they are currently inactive and are predicted to leave the institution.

===== Customer Summary Case 3 =====


This high-earning customer has an excellent credit score but currently shows low engagement with a zero balance and only one product. Despite being an inactive member, they are predicted to remain loyal to the bank.



#### **Qualitative Analysis**

- **Relevance:**  
  The summaries are well aligned with the customer data and prediction. Each profile reflects key attributes such as credit score, balance, activity status, and salary. The prediction (Stay/Churn) is also incorporated into the description.  

- **Detail & Completeness:**  
  The summaries provide sufficient information within a short format, covering main behavioral and financial aspects of the customer.  
  However, due to the strict sentence limit the level of detail is limited and does not fully capture deeper relationships between features.

- **Clarity & Readability:**  
  The outputs are clear, concise, and easy to understand. The use of simple sentences makes them highly accessible to non-technical users.  

- **Personalization:**  
  Each summary reflects the specific customer’s profile, including their engagement level, financial status, and product usage.  
  Despite this, some sentence structures are similar across cases which slightly reduces uniqueness.

- **Safety & Factual Accuracy:**  
  The summaries are mostly grounded in the provided features and prediction, avoiding major hallucinations.  
  However, minor generalizations ( assumptions about loyalty or behavior) are made without explicit evidence from the data.



#### **Quantitative Analysis**

- **Response Length:**  
  Each case produces 2–3 sentences, maintaining consistent and controlled output length.

- **Keyword Alignment:**  
  Relevant terms such as “credit score,” “balance,” “engagement,” “salary,” and “product” are consistently used, showing strong alignment with the banking domain.

- **Readability Level:**  
  Flesch Reading Ease ≈ 75, Flesch-Kincaid Grade ≈ 6  
  The outputs are very easy to read due to short sentences and simple vocabulary.  
  However, the simplicity limits analytical depth.

- **User Preference Simulation:**  
  Users who prefer quick and clear summaries would favor this prompt, as it provides a concise overview of the customer profile.  
  However, users seeking detailed explanations or actionable insights may find it too brief.


#### **Overall Observation**

The customer summary prompt produces concise, clear, and well-structured outputs that effectively describe customer profiles. The summaries are aligned with the input data and are easy to understand, making them suitable for quick insights and reporting.

However, the strict brevity limits the depth of analysis, and some generalization reduces precision. Despite these limitations the prompt is effective for generating simple and readable customer overviews.


---

## **Best Prompt Selection & Justification**

---

## **Integration Plan for Final System**


---

## **Ethical Considerations & Limitations**